# Trakπ — OTP baseline vs directTransitSearch (rail), full Norway, prod testset

Runs the production-shaped comparison of two on-disk OTP builds against a **full-Norway** graph and a
**prod-sampled testset**, driven from the trakpi **library**: `Tester` writes each build's results to
disk (`FileResultsWriter`), then `Comparator` scores the modified build against the baseline in a
second pass. Nothing is held in memory, so it scales to tens of thousands of requests.

Prepare the testset first with the CLI (`trakpi testset prepare --version prod-50k`) —
this notebook consumes it from disk. Runs on a host with ~32 GB RAM (full Norway). Baseline serves on
**8084**, modified on **8086**.

_The sequential `Tester` makes the two test passes slow at 50k; validate at ~1k first. Concurrency is
a later change._

## 1. Dependencies

The core jars plus `storage-file` for `FileResultsWriter`/`FileResultsReader`. Build them first with
`mvn -f core/pom.xml install && mvn -f storage/file/pom.xml install`.

In [1]:
@file:DependsOn("/workspace/trakpi/core/common/target/common-1.0.0-SNAPSHOT.jar")
@file:DependsOn("/workspace/trakpi/core/tester/target/tester-1.0.0-SNAPSHOT.jar")
@file:DependsOn("/workspace/trakpi/core/orchestrator/target/orchestrator-1.0.0-SNAPSHOT.jar")
@file:DependsOn("/workspace/trakpi/storage/file/target/storage-file-1.0.0-SNAPSHOT.jar")

In [2]:
import java.net.URI
import java.net.http.HttpClient
import java.net.http.HttpRequest
import java.net.http.HttpResponse.BodyHandlers
import java.io.File
import java.nio.file.Files
import java.nio.file.Path
import java.nio.file.StandardCopyOption
import java.time.Duration
import java.time.Instant
import java.util.concurrent.TimeUnit
import java.util.zip.ZipInputStream
import kotlinx.serialization.json.*

import org.opentripplanner.trakpi.common.PlannerVersion
import org.opentripplanner.trakpi.common.TestsetVersion
import org.opentripplanner.trakpi.orchestrator.PlannerOrchestrator
import org.opentripplanner.trakpi.orchestrator.PlannerStatus
import org.opentripplanner.trakpi.tester.Comparator
import org.opentripplanner.trakpi.tester.DirectoryRequestFileLoader
import org.opentripplanner.trakpi.tester.Tester
import org.opentripplanner.trakpi.tester.spi.*
import org.opentripplanner.trakpi.tester.spi.kpi.*
import org.opentripplanner.trakpi.storage.file.FileResultsReader
import org.opentripplanner.trakpi.storage.file.FileResultsWriter

## 2. Configuration

In [48]:
val CLIENT_NAME = "entur-trakpi-notebook"
val baselineJar = "/workspace/otp/otp-shaded/target/otp-baseline.jar"
val modifiedJar = "/workspace/otp/otp-shaded/target/otp-directtransit-train.jar"
val graphDir    = "/workspace/otp-data/otp/norway"
val testsetDir  = "/workspace/trakpi/local-testsets/transmodel/prod-50k"   // output of `testset prepare`
val resultsDir  = "/workspace/trakpi/results"
val testset     = TestsetVersion("prod-50k")
val baseV = PlannerVersion("base")
val modV  = PlannerVersion("mod")
val reportPath = "/workspace/trakpi/local-results/results-parsed/dts-train-deviations.md"
val dtsJar = "/workspace/otp/otp-shaded/target/otp-directtransit.jar"   // config-driven build
val coachV = PlannerVersion("coach")
val coachReportPath = "/workspace/trakpi/local-results/results-parsed/dts-train-coach-deviations.md"

## 3. Planner and request loader

Requests come from the prepared testset files (real prod `trip` queries, already carrying the fields
the KPIs need). The loader parses each `{query, variables}` body; the planner POSTs it. No
`itineraryFilters: listAll` — production replication measures the real user-facing (filtered) result.

In [4]:
data class GraphQlRequest(val query: String, val variables: JsonElement? = null) : TravelPlannerRequest

class OtpRequestLoader : RequestLoader<GraphQlRequest> {
    private val lenient = Json { isLenient = true; ignoreUnknownKeys = true }
    override fun load(file: RequestFile): GraphQlRequest {
        val body = file.body.trimStart('\uFEFF')
        val obj = runCatching { lenient.parseToJsonElement(body).jsonObject }.getOrNull()
        val query = obj?.get("query")?.jsonPrimitive?.contentOrNull
        return if (query != null) GraphQlRequest(query, obj["variables"]?.takeUnless { it is JsonNull })
        else GraphQlRequest(body)
    }
}

class OtpPlanner(private val endpoint: String, private val clientName: String) : TravelPlanner<GraphQlRequest> {
    private val http = HttpClient.newBuilder().version(HttpClient.Version.HTTP_1_1).connectTimeout(Duration.ofSeconds(10)).build()
    override fun execute(request: GraphQlRequest): TravelPlannerResponse {
        val body = buildJsonObject { put("query", JsonPrimitive(request.query)); request.variables?.let { put("variables", it) } }.toString()
        val res = http.send(HttpRequest.newBuilder().uri(URI.create(endpoint))
            .header("Content-Type", "application/json").header("ET-Client-Name", clientName)
            .timeout(Duration.ofSeconds(60)).POST(HttpRequest.BodyPublishers.ofString(body)).build(), BodyHandlers.ofString())
        val json = runCatching { Json.parseToJsonElement(res.body()).jsonObject }.getOrNull()
        val data = json?.get("data") as? JsonObject; val errors = json?.get("errors") as? JsonArray
        return TravelPlannerResponse(res.body(), res.statusCode() / 100 == 2 && data != null && errors.isNullOrEmpty(),
            data?.keys?.firstOrNull() ?: "unknown", mapOf("http_status_code" to res.statusCode().toString()))
    }
}

## 4. KPIs and comparative KPIs

Per-run KPIs are computed during each test pass; the comparative KPIs (baseline as reference) are
computed by the second-pass `Comparator`. Each itinerary is fingerprinted by its legs (mode, service
journey, line, stops, aimed times), as in trakpi's reference comparison.

In [34]:
fun TravelPlannerResponse.otpTrip(): JsonObject? =
    runCatching { Json.parseToJsonElement(raw).jsonObject["data"]?.jsonObject?.get("trip")?.jsonObject }.getOrNull()
fun TravelPlannerResponse.patterns(): JsonArray? = otpTrip()?.get("tripPatterns") as? JsonArray

class ItineraryCount : KPICalculator {
    override fun calculate(r: TravelPlannerResponse) = r.patterns()?.let { Kpi("itineraryCount", it.size.toDouble()) }
}
class RoutingTimeMs : KPICalculator {
    override fun calculate(r: TravelPlannerResponse): Kpi? {
        val n = r.otpTrip()?.get("debugOutput")?.jsonObject?.get("totalTime")?.jsonPrimitive?.longOrNull ?: return null
        return Kpi("routingTimeMs", n / 1_000_000.0)
    }
}
class SuccessKpi : KPICalculator {
    override fun calculate(r: TravelPlannerResponse) = Kpi("success", if (r.success) 1.0 else 0.0)
}

// --- itinerary fingerprint (exact, both OTP) ---
fun hhmm(iso: String?): String = if (iso != null && iso.length >= 16) iso.substring(11, 16) else "--:--"
fun str(o: JsonObject, k: String) = o[k]?.jsonPrimitive?.contentOrNull
fun nested(o: JsonObject, obj: String, k: String) = (o[obj] as? JsonObject)?.get(k)?.jsonPrimitive?.contentOrNull ?: ""
fun fingerprints(r: TravelPlannerResponse): List<List<String>> =
    (r.patterns()).orEmpty().map { it.jsonObject }.map { p ->
        (p["legs"] as? JsonArray).orEmpty().map { it.jsonObject }.map { l ->
            listOf(str(l, "mode") ?: "?", nested(l, "serviceJourney", "id"), nested(l, "line", "publicCode"),
                nested(l, "fromPlace", "name"), nested(l, "toPlace", "name"),
                hhmm(str(l, "aimedStartTime")), hhmm(str(l, "aimedEndTime"))).joinToString("|")
        }
    }

class ItineraryCountMatchesBaseline : ComparativeKPICalculator {
    override fun calculate(subject: TravelPlannerResponse, reference: TravelPlannerResponse): Kpi? {
        val s = subject.patterns() ?: return null; val r = reference.patterns() ?: return null
        return Kpi("itineraryCountMatchesReference", if (s.size == r.size) 1.0 else 0.0)
    }
}
class ItinerariesMatchBaseline : ComparativeKPICalculator {
    override fun calculate(subject: TravelPlannerResponse, reference: TravelPlannerResponse): Kpi? {
        subject.otpTrip() ?: return null; reference.otpTrip() ?: return null
        return Kpi("itinerariesMatchReference", if (fingerprints(subject) == fingerprints(reference)) 1.0 else 0.0)
    }
}

## 5. Data prep and orchestrator

`downloadNetex` fetches the current full-Norway export; `buildGraph` builds the graph two-phase
(street, then transit) with the baseline jar. `LocalOtpOrchestrator` serves a jar and polls
`serverInfo`; `start` reuses an instance already serving on its port.

In [6]:
fun sh(vararg cmd: String, dir: File? = null) {
    val p = ProcessBuilder(*cmd).directory(dir).redirectErrorStream(true)
        .redirectOutput(ProcessBuilder.Redirect.appendTo(File("prepare.log"))).start()
    check(p.waitFor() == 0) { "command failed: ${cmd.joinToString(" ")} - see prepare.log" }
}

fun otpResponds(endpoint: String, clientName: String): Boolean = try {
    val http = HttpClient.newBuilder().version(HttpClient.Version.HTTP_1_1).connectTimeout(Duration.ofSeconds(5)).build()
    val res = http.send(HttpRequest.newBuilder().uri(URI.create(endpoint))
        .header("Content-Type", "application/json").header("ET-Client-Name", clientName).timeout(Duration.ofSeconds(10))
        .POST(HttpRequest.BodyPublishers.ofString("""{"query":"{ serverInfo { version } }"}""")).build(), BodyHandlers.ofString())
    val data = Json.parseToJsonElement(res.body()).jsonObject["data"] as? JsonObject
    res.statusCode() / 100 == 2 && data?.get("serverInfo") != null
} catch (e: Exception) { false }

class LocalOtpOrchestrator(val otpJar: String, val graphDir: String, val port: Int,
    val heap: String = "-Xmx20g", val clientName: String = CLIENT_NAME, val logFile: String = "otp-$port.log") : PlannerOrchestrator {
    val endpoint = "http://localhost:$port/otp/transmodel/v3"
    private var process: Process? = null

    // prepare(): fetch fresh full-Norway NeTEx and build the graph this orchestrator serves. args picks
    // the steps: "download", "build", or both (default). The graph is shared by both builds (same
    // ser.ver.id), so run this once, e.g. baseline.prepare(baseV, null).
    override fun prepare(version: PlannerVersion, args: String?) {
        val steps = args?.split(",")?.map { it.trim() }?.toSet() ?: setOf("download", "build")
        if ("download" in steps) downloadNetex()
        if ("build" in steps) buildGraph()
    }
    private fun downloadNetex() {
        val netexDir = Path.of(graphDir, "netex")
        Files.createDirectories(netexDir)
        // Clear any prior export first: files for lines dropped since the last export would otherwise
        // linger and dangle service references, failing the transit build with a service-code NPE.
        Files.newDirectoryStream(netexDir).use { stream -> stream.forEach { Files.delete(it) } }
        val zip = Path.of(graphDir, "_norway-netex.zip")
        println("[prepare] downloading full Norway NeTEx (large) ...")
        val http = HttpClient.newBuilder().followRedirects(HttpClient.Redirect.NORMAL).connectTimeout(Duration.ofSeconds(30)).build()
        val res = http.send(HttpRequest.newBuilder(URI.create("https://storage.googleapis.com/marduk-production/outbound/netex/rb_norway-aggregated-netex.zip")).GET().build(), BodyHandlers.ofInputStream())
        check(res.statusCode() / 100 == 2) { "download failed: HTTP ${res.statusCode()}" }
        res.body().use { Files.copy(it, zip, StandardCopyOption.REPLACE_EXISTING) }
        println("[prepare] extracting ...")
        ZipInputStream(Files.newInputStream(zip)).use { zis ->
            var e = zis.nextEntry
            while (e != null) {
                if (!e.isDirectory) Files.copy(zis, netexDir.resolve(Path.of(e.name).fileName), StandardCopyOption.REPLACE_EXISTING)
                e = zis.nextEntry
            }
        }
        Files.deleteIfExists(zip)
        println("[prepare] netex ready (${netexDir.toFile().listFiles()?.size ?: 0} files)")
    }
    private fun buildGraph() {
        println("[prepare] building street graph ...")
        sh("java", heap, "-jar", otpJar, "--buildStreet", "--save", graphDir)
        println("[prepare] building transit graph ...")
        sh("java", heap, "-jar", otpJar, "--loadStreet", "--save", graphDir)
        println("[prepare] graph built -> $graphDir/graph.obj")
    }

    override fun start(version: PlannerVersion, args: String?) {
        if (otpResponds(endpoint, clientName)) { println("[otp:$port] already serving - reusing"); return }
        check(process == null) { "OTP already launched on $port" }
        println("[otp:$port] launching ${File(otpJar).name} --load $graphDir")
        process = ProcessBuilder("java", heap, "-jar", otpJar, "--load", graphDir, "--port", "$port")
            .redirectErrorStream(true).redirectOutput(ProcessBuilder.Redirect.appendTo(File(logFile))).start()
    }
    override fun stop(version: PlannerVersion) {
        process?.let { it.destroy(); if (!it.waitFor(30, TimeUnit.SECONDS)) it.destroyForcibly() }; process = null
    }
    override fun status(version: PlannerVersion): PlannerStatus {
        val p = process
        return when {
            otpResponds(endpoint, clientName) -> PlannerStatus.READY
            p != null && !p.isAlive -> PlannerStatus.FAILED
            p != null -> PlannerStatus.STARTING
            else -> PlannerStatus.NOT_RUNNING
        }
    }
    fun awaitReady(version: PlannerVersion, attempts: Int = 120, interval: Duration = Duration.ofSeconds(5)): PlannerStatus {
        repeat(attempts) {
            when (val s = status(version)) {
                PlannerStatus.READY, PlannerStatus.FAILED, PlannerStatus.NOT_RUNNING -> return s
                PlannerStatus.STARTING -> Thread.sleep(interval.toMillis())
            }
        }
        return status(version)
    }
}

## 6. Prepare (one-time): fresh NeTEx + full graph

`prepare()` on the orchestrator fetches the current full-Norway NeTEx and builds the graph two-phase
— the SPI's data step. The graph is shared by both builds (same ser.ver.id), so run it once here on
the baseline orchestrator. Slow and heap-hungry; comment out after. Watch `prepare.log`. Pass
`"build"` to skip the download, or `"download"` to skip the build.

In [8]:
val baseline = LocalOtpOrchestrator(baselineJar, graphDir, port = 8084)

// Run once (fresh NeTEx + graph), then comment out:
baseline.prepare(baseV, null)

[prepare] downloading full Norway NeTEx (large) ...
[prepare] extracting ...
[prepare] netex ready (4832 files)
[prepare] building street graph ...
[prepare] building transit graph ...
[prepare] graph built -> /workspace/otp-data/otp/norway/graph.obj


## 7. Test the baseline build

Serve the baseline, run the whole testset through `Tester`, writing per-run results to
`$resultsDir/base`. Re-run the poll line until READY. The test cell is slow at 50k (sequential).

In [19]:
baseline.start(baseV, null)
baseline.awaitReady(baseV)

[otp:8084] launching otp-baseline.jar --load /workspace/otp-data/otp/norway


READY

In [20]:
check(baseline.status(baseV) == PlannerStatus.READY) { "baseline not READY - re-run the cell above" }
Tester(
    run = RunMetadata.create(version = baseV, application = "otp", startedAt = Instant.now(), testsetVersion = testset, runId = "base"),
    requestFileLoader = DirectoryRequestFileLoader(Path.of(testsetDir)),
    requestLoader = OtpRequestLoader(),
    travelPlanner = OtpPlanner(baseline.endpoint, CLIENT_NAME),
    kpiCalculators = listOf(ItineraryCount(), RoutingTimeMs(), SuccessKpi()),
    resultsWriter = FileResultsWriter(Path.of(resultsDir)),
).run()
baseline.stop(baseV)
"baseline done -> $resultsDir/base"

0% complete. 100/49363 requests processed.
0% complete. 200/49363 requests processed.
0% complete. 300/49363 requests processed.
0% complete. 400/49363 requests processed.
1% complete. 500/49363 requests processed.
1% complete. 600/49363 requests processed.
1% complete. 700/49363 requests processed.
1% complete. 800/49363 requests processed.
1% complete. 900/49363 requests processed.
2% complete. 1000/49363 requests processed.
2% complete. 1100/49363 requests processed.
2% complete. 1200/49363 requests processed.
2% complete. 1300/49363 requests processed.
2% complete. 1400/49363 requests processed.
3% complete. 1500/49363 requests processed.
3% complete. 1600/49363 requests processed.
3% complete. 1700/49363 requests processed.
3% complete. 1800/49363 requests processed.
3% complete. 1900/49363 requests processed.
4% complete. 2000/49363 requests processed.
4% complete. 2100/49363 requests processed.
4% complete. 2200/49363 requests processed.
4% complete. 2300/49363 requests processe

baseline done -> /workspace/trakpi/results/base

In [21]:
val modified = LocalOtpOrchestrator(modifiedJar, graphDir, port = 8086)
modified.start(modV, null)
modified.awaitReady(modV)

[otp:8086] launching otp-directtransit-train.jar --load /workspace/otp-data/otp/norway


READY

In [22]:
check(modified.status(modV) == PlannerStatus.READY) { "modified not READY - re-run the cell above" }
Tester(
    run = RunMetadata.create(version = modV, application = "otp", startedAt = Instant.now(), testsetVersion = testset, runId = "mod"),
    requestFileLoader = DirectoryRequestFileLoader(Path.of(testsetDir)),
    requestLoader = OtpRequestLoader(),
    travelPlanner = OtpPlanner(modified.endpoint, CLIENT_NAME),
    kpiCalculators = listOf(ItineraryCount(), RoutingTimeMs(), SuccessKpi()),
    resultsWriter = FileResultsWriter(Path.of(resultsDir)),
).run()
modified.stop(modV)
"modified done -> $resultsDir/mod"

0% complete. 100/49363 requests processed.
0% complete. 200/49363 requests processed.
0% complete. 300/49363 requests processed.
0% complete. 400/49363 requests processed.
1% complete. 500/49363 requests processed.
1% complete. 600/49363 requests processed.
1% complete. 700/49363 requests processed.
1% complete. 800/49363 requests processed.
1% complete. 900/49363 requests processed.
2% complete. 1000/49363 requests processed.
2% complete. 1100/49363 requests processed.
2% complete. 1200/49363 requests processed.
2% complete. 1300/49363 requests processed.
2% complete. 1400/49363 requests processed.
3% complete. 1500/49363 requests processed.
3% complete. 1600/49363 requests processed.
3% complete. 1700/49363 requests processed.
3% complete. 1800/49363 requests processed.
3% complete. 1900/49363 requests processed.
4% complete. 2000/49363 requests processed.
4% complete. 2100/49363 requests processed.
4% complete. 2200/49363 requests processed.
4% complete. 2300/49363 requests processe

modified done -> /workspace/trakpi/results/mod

## 9. Compare (second pass, offline)

`Comparator` reads both runs' responses from disk, scores each modified response against the baseline,
and merges the comparative KPIs (`itineraryCountMatchesReference`, `itinerariesMatchReference`) into
the modified run's result files. No OTP needed.

In [23]:
Comparator(
    run = RunMetadata.create(version = modV, application = "otp", startedAt = Instant.now(), testsetVersion = testset, referenceVersion = baseV, runId = "mod"),
    referenceVersion = baseV,
    resultsReader = FileResultsReader(Path.of(resultsDir)),
    comparativeKpiCalculators = listOf(ItineraryCountMatchesBaseline(), ItinerariesMatchBaseline()),
    resultsWriter = FileResultsWriter(Path.of(resultsDir)),
).run()

Comparing mod against base on testset prod-50k: 49363 subject and 49363 reference response(s).
Wrote comparative KPIs for 14372 request(s).


## 10. Aggregate — production impact, rail vs non-rail

Reads the modified run's result files: how many production requests the feature changes, split by
whether the journey involves rail. Non-rail should be ~all-match (the rail-only restriction held);
rail is the real user-facing impact.

In [27]:
fun rawOf(dir: String, req: String): TravelPlannerResponse? {
    val f = File("$resultsDir/$dir/$req.json"); if (!f.exists()) return null
    val o = Json.parseToJsonElement(f.readText()).jsonObject
    return TravelPlannerResponse(o["rawResponse"]?.jsonPrimitive?.contentOrNull ?: "",
        o["success"]?.jsonPrimitive?.booleanOrNull ?: false, o["method"]?.jsonPrimitive?.contentOrNull ?: "", emptyMap())
}
fun involvesRail(r: TravelPlannerResponse?): Boolean =
    r?.patterns().orEmpty().any { p -> (p.jsonObject["legs"] as? JsonArray).orEmpty().any { (it.jsonObject["mode"]?.jsonPrimitive?.contentOrNull) == "rail" } }

val modFiles = File("$resultsDir/mod").listFiles { f -> f.name.endsWith(".json") }.orEmpty()
var railMatch = 0; var railDiffer = 0; var otherMatch = 0; var otherDiffer = 0; var noKpi = 0
val differing = mutableListOf<String>()
for (f in modFiles) {
    val o = Json.parseToJsonElement(f.readText()).jsonObject
    val m = (o["kpis"] as? JsonObject)?.get("itinerariesMatchReference")?.jsonPrimitive?.doubleOrNull
    if (m == null) { noKpi++; continue }
    val req = f.name.removeSuffix(".json")
    val rail = involvesRail(rawOf("base", req)) || involvesRail(rawOf("mod", req))
    if (m == 1.0) { if (rail) railMatch++ else otherMatch++ }
    else { if (rail) railDiffer++ else otherDiffer++; differing += req }
}
val total = modFiles.size
HTML("<table><tr><th></th><th>match</th><th>differ</th></tr>" +
    "<tr><td>rail journeys</td><td>$railMatch</td><td>$railDiffer</td></tr>" +
    "<tr><td>non-rail</td><td>$otherMatch</td><td>$otherDiffer</td></tr>" +
    "</table><p>$total requests total; ${differing.size} changed by directTransitSearch" +
    "${if (noKpi > 0) "; $noKpi had no comparable response" else ""}.</p>")

,match,differ
rail journeys,758,209
non-rail,13355,50


## 11. Drill down into changed requests

For a few requests the feature changed, the baseline vs modified itineraries (mode/line + aimed
times), so you can see what directTransitSearch added.

In [33]:
fun render(r: TravelPlannerResponse?): List<String> =
    r?.patterns().orEmpty().map { p ->
        val legs = (p.jsonObject["legs"] as? JsonArray).orEmpty().map { it.jsonObject }
        legs.joinToString("  ->  ") { l ->
            val mode = str(l, "mode"); val line = nested(l, "line", "publicCode")
            if (mode == "foot") "walk" else "${line.ifEmpty { mode }} ${hhmm(str(l, "aimedStartTime"))}-${hhmm(str(l, "aimedEndTime"))}"
        }
    }

val sb = StringBuilder()
for (req in differing.take(5)) {
    sb.append("<h4>$req</h4><b>baseline</b><pre>")
    render(rawOf("base", req)).forEachIndexed { i, s -> sb.append("%2d. %s\n".format(i + 1, s)) }
    sb.append("</pre><b>modified</b><pre>")
    render(rawOf("mod", req)).forEachIndexed { i, s -> sb.append("%2d. %s\n".format(i + 1, s)) }
    sb.append("</pre>")
}
HTML(sb.toString())

org.jetbrains.kotlinx.jupyter.exceptions.ReplCompilerException: at Cell In[33], line 5, column 24: Cannot access 'str': it is private in 'Line_7_jupyter'
at Cell In[33], line 5, column 51: Cannot access 'nested': it is private in 'Line_7_jupyter'
at Cell In[33], line 6, column 78: Cannot access 'str': it is private in 'Line_7_jupyter'
at Cell In[33], line 6, column 112: Cannot access 'str': it is private in 'Line_7_jupyter'

## 12. Deviation report (markdown)

Writes a human-readable diff of every search whose itineraries changed, to `reportPath`. A `+` line
is an itinerary the candidate added, `-` one it dropped, blank is unchanged. Itineraries are matched
fuzzily on transit line + departure/arrival minute, so walk-only and sub-minute differences are
ignored. Blocks are sorted by how much the search changed.

In [45]:
// Writes a human-readable deviation report of a subject run vs a reference run (both under resultsDir).
// "off" = reference itinerary count, "on" = subject count; "+" lines are itineraries the subject added,
// "-" ones it dropped. Matching is fuzzy: transit line + dep/arr minute (walk-only & sub-minute ignored).
fun writeDeviationReport(referenceDir: String, subjectDir: String, outPath: String, title: String) {
    val refFiles = File("$resultsDir/$referenceDir").listFiles { f -> f.name.endsWith(".json") }.orEmpty()
    val changed = mutableListOf<Changed>()
    var nTrip = 0; var totAdd = 0; var totRem = 0
    for (f in refFiles) {
        val req = f.name.removeSuffix(".json")
        val b = rawOf(referenceDir, req) ?: continue
        if (b.method != "trip" || b.patterns() == null) continue
        nTrip++
        val m = rawOf(subjectDir, req) ?: continue
        val baseSigs = LinkedHashMap<List<String>, JsonObject>(); val modSigs = LinkedHashMap<List<String>, JsonObject>()
        b.patterns().orEmpty().forEach { baseSigs.putIfAbsent(devSig(it.jsonObject), it.jsonObject) }
        m.patterns().orEmpty().forEach { modSigs.putIfAbsent(devSig(it.jsonObject), it.jsonObject) }
        val added = modSigs.keys.filter { it !in baseSigs }; val removed = baseSigs.keys.filter { it !in modSigs }
        if (added.isEmpty() && removed.isEmpty()) continue
        totAdd += added.size; totRem += removed.size
        val rows = mutableListOf<Triple<String, String, String>>()
        baseSigs.forEach { (s, p) -> rows += Triple(startOf(p), if (s in removed) "-" else " ", renderFull(p)) }
        modSigs.forEach { (s, p) -> if (s !in baseSigs) rows += Triple(startOf(p), "+", renderFull(p)) }
        rows.sortBy { it.first }
        val first = b.patterns().orEmpty().firstOrNull()?.jsonObject
        val frm = first?.let { legsOf(it).firstOrNull()?.let { l -> nested(l, "fromPlace", "name") } }.orEmpty().ifEmpty { "?" }
        val to = first?.let { legsOf(it).lastOrNull()?.let { l -> nested(l, "toPlace", "name") } }.orEmpty().ifEmpty { "?" }
        val depart = rows.firstOrNull()?.first?.let { "${it.take(10)} ${hhmm(it)}" } ?: ""
        changed += Changed(req, frm, to, depart, modSigs.size, baseSigs.size, added.size, removed.size, rows)
    }
    changed.sortByDescending { it.add + it.rem }
    val md = StringBuilder()
    md.appendLine("# $title").appendLine()
    md.appendLine("Reference run `$referenceDir` vs subject run `$subjectDir`, testset `${testset.value}`.").appendLine()
    md.appendLine("Each block: a `+` line is an itinerary the subject added, a `-` line one it dropped, a blank prefix means unchanged. Itineraries are matched on transit line + departure/arrival minute, ignoring walk-only and sub-minute differences.").appendLine()
    md.appendLine("## Summary").appendLine()
    md.appendLine("- trip requests compared: **$nTrip**")
    md.appendLine("- searches whose itineraries changed: **${changed.size}** (${"%.1f".format(100.0 * changed.size / maxOf(nTrip, 1))}%)")
    md.appendLine("- itineraries added by subject: **$totAdd**")
    md.appendLine("- itineraries removed by subject: **$totRem**").appendLine()
    md.appendLine("## Changed searches").appendLine()
    for (c in changed) {
        md.appendLine("### ${c.frm} → ${c.to}").appendLine()
        md.appendLine("`${c.rid}` · delta ${if (c.on - c.off >= 0) "+" else ""}${c.on - c.off} (on ${c.on} / off ${c.off}) [+${c.add} -${c.rem}] · depart ${c.depart}").appendLine()
        md.appendLine("```")
        c.rows.forEach { (_, pref, text) -> md.appendLine("$pref $text") }
        md.appendLine("```").appendLine()
    }
    File(outPath).parentFile?.mkdirs()
    File(outPath).writeText(md.toString())
    println("Wrote $outPath — $nTrip trip requests, ${changed.size} changed (+$totAdd / -$totRem).")
}



In [38]:
// --- full, human-readable itinerary rendering + fuzzy signature (reuses helpers from earlier cells) ---
fun legsOf(p: JsonObject) = (p["legs"] as? JsonArray).orEmpty().map { it.jsonObject }
fun isTransit(l: JsonObject) = (l["line"] as? JsonObject) != null
fun headSign(l: JsonObject) =
    ((l["fromEstimatedCall"] as? JsonObject)?.get("destinationDisplay") as? JsonObject)?.get("frontText")?.jsonPrimitive?.contentOrNull ?: ""
fun legLine(l: JsonObject): String {
    val frm = nested(l, "fromPlace", "name").ifEmpty { "?" }
    val to = nested(l, "toPlace", "name").ifEmpty { "?" }
    val start = hhmm(str(l, "aimedStartTime")); val end = hhmm(str(l, "aimedEndTime"))
    val desc = if (isTransit(l)) "${nested(l, "line", "publicCode")} ${headSign(l)}".trim() else "walk"
    return "$desc from $frm ($start) -> $to ($end)"
}
fun renderFull(p: JsonObject): String {
    val mins = Math.round((p["duration"]?.jsonPrimitive?.doubleOrNull ?: 0.0) / 60.0)
    return "(${mins}m) " + legsOf(p).joinToString(" -> ") { legLine(it) }
}
// signature: transit legs only, matched on line publicCode + dep/arr minute (walk & sub-minute ignored)
fun devSig(p: JsonObject): List<String> =
    legsOf(p).filter { isTransit(it) }.map { "${nested(it, "line", "publicCode")}|${hhmm(str(it, "aimedStartTime"))}|${hhmm(str(it, "aimedEndTime"))}" }
fun startOf(p: JsonObject) = legsOf(p).firstOrNull()?.let { str(it, "aimedStartTime") } ?: ""

data class Changed(val rid: String, val frm: String, val to: String, val depart: String,
    val on: Int, val off: Int, val add: Int, val rem: Int, val rows: List<Triple<String, String, String>>)

writeDeviationReport("base", "mod", reportPath, "DirectTransitSearch (train-only) — full-Norway prod replication")
HTML("Wrote <code>$reportPath</code> — see printout for counts.")

Wrote /workspace/trakpi/local-results/results-parsed/dts-train-deviations.md 14372 trip requests, 219 changed (+830 / -211 itineraries).

## 13. Third comparison: train + long-distance coach (config-driven)

Runs the config-driven jar (`dtsJar`) with `directTransitSearch` enabled for **rail + long-distance coach**
via `router-config.json` (`routingDefaults.directTransitSearch.modes = ["RAIL", "COACH"]`),
over the same `prod-50k` testset. Reuses the existing `base` results as the reference, so only this
one new run is needed.

Note: writing this config into the shared `graphDir` enables the feature for anything served from it.
`base`/`mod` reuse their stored results and should not be re-run against this config without resetting
`router-config.json` back to `{}`.

In [49]:
// directTransitSearch is configured in router-config.json (routingDefaults). Enable it for the
// config-driven jar, restricted to rail + long-distance coach, before serving.
fun writeDirectTransitConfig(modes: List<String>) {
    val modesJson = modes.joinToString(", ") { "\"$it\"" }
    File("$graphDir/router-config.json").writeText(
        """
        {
          "routingDefaults": {
            "directTransitSearch": {
              "enabled": true,
              "modes": [$modesJson]
            }
          }
        }
        """.trimIndent()
    )
}
writeDirectTransitConfig(listOf("RAIL", "COACH"))

val coach = LocalOtpOrchestrator(dtsJar, graphDir, port = 8088)
coach.start(coachV, null)
coach.awaitReady(coachV)

[otp:8088] launching otp-directtransit.jar --load /workspace/otp-data/otp/norway


READY

In [50]:
check(coach.status(coachV) == PlannerStatus.READY) { "coach not READY - re-run the cell above" }
Tester(
    run = RunMetadata.create(version = coachV, application = "otp", startedAt = Instant.now(), testsetVersion = testset, runId = "coach"),
    requestFileLoader = DirectoryRequestFileLoader(Path.of(testsetDir)),
    requestLoader = OtpRequestLoader(),
    travelPlanner = OtpPlanner(coach.endpoint, CLIENT_NAME),
    kpiCalculators = listOf(ItineraryCount(), RoutingTimeMs(), SuccessKpi()),
    resultsWriter = FileResultsWriter(Path.of(resultsDir)),
).run()
coach.stop(coachV)
"coach done -> $resultsDir/coach"

0% complete. 100/49363 requests processed.
0% complete. 200/49363 requests processed.
0% complete. 300/49363 requests processed.
0% complete. 400/49363 requests processed.
1% complete. 500/49363 requests processed.
1% complete. 600/49363 requests processed.
1% complete. 700/49363 requests processed.
1% complete. 800/49363 requests processed.
1% complete. 900/49363 requests processed.
2% complete. 1000/49363 requests processed.
2% complete. 1100/49363 requests processed.
2% complete. 1200/49363 requests processed.
2% complete. 1300/49363 requests processed.
2% complete. 1400/49363 requests processed.
3% complete. 1500/49363 requests processed.
3% complete. 1600/49363 requests processed.
3% complete. 1700/49363 requests processed.
3% complete. 1800/49363 requests processed.
3% complete. 1900/49363 requests processed.
4% complete. 2000/49363 requests processed.
4% complete. 2100/49363 requests processed.
4% complete. 2200/49363 requests processed.
4% complete. 2300/49363 requests processe

coach done -> /workspace/trakpi/results/coach

In [51]:
// Score coach against the baseline (train+coach impact), and against mod (incremental effect
// of adding long-distance coaches on top of rail).
listOf(baseV to "base", modV to "mod").forEach { (ref, refId) ->
    Comparator(
        run = RunMetadata.create(version = coachV, application = "otp", startedAt = Instant.now(), testsetVersion = testset, referenceVersion = ref, runId = "coach"),
        referenceVersion = ref,
        resultsReader = FileResultsReader(Path.of(resultsDir)),
        comparativeKpiCalculators = listOf(ItineraryCountMatchesBaseline(), ItinerariesMatchBaseline()),
        resultsWriter = FileResultsWriter(Path.of(resultsDir)),
    ).run()
}
"compared coach vs base and vs mod"

Comparing coach against base on testset prod-50k: 49363 subject and 49363 reference response(s).
Wrote comparative KPIs for 14372 request(s).
Comparing coach against mod on testset prod-50k: 49363 subject and 49363 reference response(s).
Wrote comparative KPIs for 14372 request(s).


compared coach vs base and vs mod

In [52]:
writeDeviationReport("base", "coach", coachReportPath, "DirectTransitSearch (train + long-distance coach) — full-Norway prod replication")
HTML("Wrote <code>$coachReportPath</code> — see printout for counts.")

Wrote /workspace/trakpi/local-results/results-parsed/dts-train-coach-deviations.md — 14372 trip requests, 253 changed (+903 / -234).


Wrote /workspace/trakpi/local-results/results-parsed/dts-train-coach-deviations.md — see printout for counts.